## 02 – Modelling: Detecting Credit Card Fraud

**Goal:** Train a model to spot fraudulent transactions, then estimate how much money it would save.

**What we learned in notebook 1:**
- Fraud is very rare (0.167%), so accuracy is misleading. Models are judged on how much fraud they catch (**recall**)
  and how many of their alerts are correct (**precision**).
- Fraud happens more at night, so the hour of the transaction is given to the model.

**Steps:** prepare the data → simple baseline → two models → compare → money saved → what the model relied on.

### Setup

Load the data and remove duplicates (the checks are in notebook 1). Add an `hour` column, since fraud is more common at night.

In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
#sklearn - Python’s main machine learning library
from sklearn.model_selection import train_test_split
#Splits the data into the 80% training and 20% test parts
from sklearn.dummy import DummyClassifier
#The “lazy” model that always guesses normal (baseline)
from sklearn.linear_model import LogisticRegression
#Model 1, the simple formula model
from sklearn.ensemble import RandomForestClassifier
#Model 2, the 100 voting decision trees
from sklearn.pipeline import make_pipeline
#Chains steps together, so scaling then the model happen as one
from sklearn.preprocessing import StandardScaler
#Puts all features on the same scale
from sklearn.metrics import (accuracy_score, precision_score, recall_score, average_precision_score, 
                             ConfusionMatrixDisplay #Draws the 2×2 grid of right and wrong answers
                            )

database_connection = duckdb.connect()
DATA_PATH = "../data/creditcard.csv"

#Load the CSV and remove duplicate rows
database_connection.execute(f"""
CREATE OR REPLACE TABLE transactions AS
SELECT DISTINCT * FROM read_csv_auto('{DATA_PATH}')
""")

#Add the hour of day (0-23) and turn it into a table
df = database_connection.execute("""
SELECT *, CAST(FLOOR(Time / 3600) AS INTEGER) % 24 AS hour
FROM transactions
""").df()

print(f"{len(df):,} transactions, {int(df['Class'].sum())} fraud")

283,726 transactions, 473 fraud
